In [0]:
%sql

select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables
where document_id='412320'

select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary
where Document_Id='14588888'
order by Data_Provider_ID, datafield_id asc

In [0]:
%sql
select distinct Document_Id, Data_Provider_ID, DataSource_Excel_Path, DataSource_Excel_Tab
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_excel

In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, trim, concat_ws

# Step 1 — filter to Measure rows only
df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables") 
# \
#     .filter(col("variable_qualification") == "Measure")

# Step 2 — normalize variable_definition: strip [], lowercase, collapse whitespace
df = df.withColumn(
    "variable_definition_normalized",
    regexp_replace(
        lower(trim(
            regexp_replace(col("variable_definition"), r"[\[\]]", "")
        )),
        r"\s+", " "
    )
)

# Step 3 — unique ID = Document_Id + variable_id
df = df.withColumn("unique_id", concat_ws("_", col("Document_Id"), col("variable_id")))

# Output as pandas
pdf = df.select("Document_Id", "unique_id", "variable_name","variable_qualification", "variable_definition_normalized").toPandas()
display(pdf)

In [0]:
import re
import pandas as pd

# ---- Category rules (priority order — first match wins) ----
def categorize_definition(defn):
    if pd.isna(defn) or str(defn).strip() == '':
        return 'Empty'
    d = str(defn).strip().lower()

    # 1. Sum aggregate — =sum(...)
    if re.search(r'=\s*sum\s*\(', d):
        return 'Sum Aggregate'

    # 2. Conditional — contains if(...)
    if re.search(r'\bif\s*\(', d):
        return 'Conditional (IF)'

    # 3. FX / Rate conversion — loc:eur, eur:loc, exchange rate patterns
    if re.search(r'(loc\s*:\s*eur|eur\s*:\s*loc|exchange\s*rate|fx\s*rate)', d):
        return 'FX / Rate Conversion'

    # 4. Arithmetic DR/CR — debit minus credit patterns
    if re.search(r'\bdr\b.*\bcr\b|\bcr\b.*\bdr\b', d):
        return 'Arithmetic DR/CR'

    # 5. General arithmetic — contains +, -, *, /
    if re.search(r'[+\-*/]', d):
        return 'Arithmetic'

    # 6. Anything else — simple field reference
    return 'Simple Reference'


pdf['category'] = pdf['variable_definition_normalized'].apply(categorize_definition)

# ---- Distribution summary ----
dist = pdf['category'].value_counts().rename_axis('category').reset_index(name='count')
dist['pct'] = (dist['count'] / len(pdf) * 100).round(1).astype(str) + '%'
print(f"Total rows: {len(pdf):,}")
print()
print(dist.to_string(index=False))
print()

# ---- Distinct Document_Id count per category ----
doc_dist = (
    pdf.groupby('category')['Document_Id']
    .nunique()
    .reset_index(name='distinct_doc_count')
    .sort_values('distinct_doc_count', ascending=False)
)
print("Distinct Document_Id per category:")
print(doc_dist.to_string(index=False))
print()

# ---- Full output ----
display(pdf[['unique_id', 'variable_name', 'variable_definition_normalized', 'category']].sort_values('category'))

In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, trim, concat_ws

# Step 1 — filter to Measure rows only
df_dd = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary")

# Step 2 — form DataSource by DataSource_Name + DataSource_Type
df_dd = df_dd.withColumn("DataSource", concat_ws("_", col("DataSource_Name"), col("DataSource_Type")))

# Step 3 — unique ID = Document_Id + datafield_id
df_dd = df_dd.withColumn("unique_id", concat_ws("_", col("Document_Id"), col("datafield_id")))

# Output as pandas
pdf_dd = df_dd.select(
    "Document_Id", "unique_id", "datafield_name", "datafield_datatype",
    "DataSource", "DataSource_Type"
).toPandas()
display(pdf_dd)

In [0]:
# ---- Set 1: distinct Document_Id count per Universe_FHSQL ----
set1 = (
    pdf_dd.groupby('DataSource')['Document_Id']
    .nunique()
    .reset_index(name='distinct_doc_count')
    .sort_values('distinct_doc_count', ascending=False)
)
print(f"Set 1 — Distinct Document_Id per Universe_FHSQL ({len(set1)} unique values):")
display(set1)

# ---- Analysis 1: % distribution of all unique_id by DataSource_Type ----
dist_unique_id = (
    pdf_dd.groupby('DataSource_Type')['unique_id']
    .nunique()
    .reset_index(name='unique_id_count')
)
total_unique_ids = dist_unique_id['unique_id_count'].sum()
dist_unique_id['pct_of_total'] = (dist_unique_id['unique_id_count'] / total_unique_ids * 100).round(1).astype(str) + '%'
dist_unique_id = dist_unique_id.sort_values('unique_id_count', ascending=False)
print("Analysis 1 — % distribution of all unique_id by DataSource_Type:")
display(dist_unique_id)

# ---- Analysis 2: count of documents referring data from each DataSource_Type ----
doc_count_by_type = (
    pdf_dd.groupby('DataSource_Type')['Document_Id']
    .nunique()
    .reset_index(name='distinct_doc_count')
    .sort_values('distinct_doc_count', ascending=False)
)
print("Analysis 2 — Count of documents referring data from each DataSource_Type:")
display(doc_count_by_type)

In [0]:
%sql
select case when lower(DataSource_Type)='unv' then 'Old version universe' when lower(DataSource_Type)='unx' then 'New version universe' else null end as Universt_Type , DataSource_Name as Universe_name, count(distinct Document_Id) as doc_cnt from 
dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary
where lower(DataSource_Type) in ('unv','unx')
group by DataSource_Type, DataSource_Name
order by doc_cnt desc

select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary

ALTER TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary RENAME TO dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Z_archive_webi_dataDictionary;